# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata_json = json.loads(dataset.metadata.to_json_string())

print(f"{metadata_json.get('name', '')}: {metadata_json.get('description', '')}")
print(f"Dataset Version: {metadata_json.get('version', '')}")

## 2. Data Overview
Review all available record sets and their field `@id`s as provided by the Croissant schema.

To ensure proper referencing, each entity is referenced by its `@id`.

In [ ]:
# List all available record sets in the dataset by @id
record_sets = dataset.metadata.record_sets
print(f"Total record sets found: {len(record_sets)}")
for idx, rs in enumerate(record_sets):
    print(f"[{idx}] RecordSet @id: {rs['@id']}")
    print(f"    Name: {rs.get('name', '')}")
    print(f"    Description: {rs.get('description', '')}")
    # List fields (columns) by @id
    if 'field' in rs and isinstance(rs['field'], list):
        print("    Fields (by @id):")
        for f in rs['field']:
            fid = f['@id'] if isinstance(f, dict) and '@id' in f else f
            print(f"      - {fid}")
    elif 'field' in rs:
        fid = rs['field']['@id'] if isinstance(rs['field'], dict) else rs['field']
        print(f"    Field (by @id): {fid}")
    print()

## 3. Data Extraction
Load data from each record set into a DataFrame. All references to record sets or fields are made using their `@id`.

In [ ]:
# List of record set @ids for extraction (grab all for generality)
record_set_ids = [rs['@id'] for rs in dataset.metadata.record_sets]

# Dictionary to store DataFrames
dataframes = {}

for record_set_id in record_set_ids:
    print(f"Extracting: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Columns in {record_set_id}: {df.columns.tolist()}")
    print(df.head(2))
    print("---\n")
# For further analysis, pick the first record set
main_record_set_id = record_set_ids[0] if record_set_ids else None
main_df = dataframes[main_record_set_id] if main_record_set_id else pd.DataFrame()

## 4. Exploratory Data Analysis (EDA)
Let's filter, normalize, and group data using appropriate fields. All fields are referenced by their `@id`.

Here, we choose a numeric field (e.g. protein MW or abundance if present) and a group field (e.g. a categorical field like Sample Type, if present) based on the fields listed in Section 2.

In [ ]:
# Example logic: choose first numeric field available
import numpy as np

# Try to auto-detect a numeric field in the main data frame
numeric_field_id = None
for col in main_df.columns:
    # Infer numeric if dtype or typical proteomics column name
    if main_df[col].dtype in [np.float64, np.float32, np.int64, np.int32]:
        numeric_field_id = col
        break
    # Some files have string numbers, so check convertible
    try:
        main_df[col].astype(float)
        numeric_field_id = col
        break
    except Exception:
        continue
if numeric_field_id is None:
    print("No numeric field found.")
else:
    # Ensure numeric dtype
    main_df[numeric_field_id] = pd.to_numeric(main_df[numeric_field_id], errors='coerce')
    threshold = main_df[numeric_field_id].quantile(0.80)  # Use 80th percentile as threshold

    filtered_df = main_df[main_df[numeric_field_id] > threshold]

    print(f"Filtered records with {numeric_field_id} > {threshold:.3f}:")
    print(filtered_df[[numeric_field_id]].head())

    # Normalization (Z-score)
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
        filtered_df[numeric_field_id].std()
    )
    print(f"Normalized '{numeric_field_id}' for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try to find a group field (string or categorical)
    group_field_id = None
    for col in main_df.columns:
        if main_df[col].dtype == object and len(main_df[col].unique()) < 10:
            group_field_id = col
            break

    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"\nGrouped by {group_field_id} (mean {numeric_field_id}):")
        print(grouped_df)
    else:
        print("No suitable group field found.")

## 5. Visualization
Visualize field distributions and relationships. Replace axes with chosen field `@id`s.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id:
    plt.figure(figsize=(8,4))
    sns.histplot(main_df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.tight_layout()
    plt.show()

    if group_field_id:
        plt.figure(figsize=(8,5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=main_df)
        plt.title(f"Boxplot of {numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=30)
        plt.tight_layout()
        plt.show()

## 6. Conclusion
We have demonstrated loading, exploring, and performing basic EDA on a Croissant-compliant dataset using `mlcroissant`.

- All entities were referenced using their `@id` for consistent interoperability.
- The record set(s) and their fields were dynamically discovered and analyzed.
- Data normalization, filtering, and basic grouping were performed on selected fields.
- Example visualizations highlight distribution and categorical comparisons.

To extend this notebook, select other record sets or adjust processing logic based on your research question or analysis needs.